In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

### Data Reading From Source

In [0]:
df = spark.sql("select * from travel_journal_catalog.silver.google_maps_address_silver")

In [0]:
df.display()

address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day
"1-1 Honmaru, Naka Ward, Nagoya, Aichi 460-0031, Japan",1,1,Japan,2026-06-23T21:25:29.26092Z,null,false,9,35.1848636,136.899927,Honmaru Palace,Naka Ward,ChIJleKGf8t2A2ARY_ZkNA5RTo8,460-0031,Aichi,Aichi,2026-06-24,2026,6,24
"830 Smith Court, Kristiansand, Norway",85977,Kristiansand,Norway,2026-07-20T18:41:30.405255Z,North Crystal,false,820,58.14671,7.9956,Golden Mccarthy Bazaar,null,SEED_D66A3D33A81D3720244D,null,null,null,2026-07-21,2026,7,21
"8551 Cody Fork Suite 501, Valencia, Spain",98738,Valencia,Spain,2026-07-20T18:41:30.526168Z,South Andreamouth,false,821,39.47391,-0.37966,Old Hodges Tower,null,SEED_92F79A98373BC16A5770,53545,null,null,2026-07-21,2026,7,21
"18153 Berger Fork, Phnom Penh, Cambodia",null,Phnom Penh,Cambodia,2026-07-20T18:41:30.644969Z,Sarahtown,false,822,11.56245,104.91601,Royal Wade Bazaar,side,SEED_F2E6E20A2A2E3FF0535E,23335,null,null,2026-07-21,2026,7,21
"1125 Selena Plaza, Durban, South Africa",null,Durban,South Africa,2026-07-20T18:41:30.765536Z,East Christine,false,823,-29.8579,31.0292,Hidden Peterson Pier,town,SEED_CFFD589432E71AED70A2,64966,null,null,2026-07-21,2026,7,21
"614 Hardy Walks, Callao, Peru",null,Callao,Peru,2026-07-20T18:41:30.881142Z,Port Kimberlystad,false,824,-12.05659,-77.11814,Golden May Harbor,view,SEED_E681890FDE3792E1C646,null,null,null,2026-07-21,2026,7,21
"042 Johnson Shoals, Budapest XVII. kerület, Hungary",null,Budapest XVII. kerület,Hungary,2026-07-20T18:41:30.995516Z,Angelville,false,825,47.47997,19.25388,Big Bailey Market,furt,SEED_D5CC32B458E07913F8FC,45535,null,null,2026-07-21,2026,7,21
"9757 Joanne Heights Apt. 555, Invercargill, New Zealand",357,Invercargill,New Zealand,2026-07-20T18:41:31.113575Z,Owensfurt,false,826,-46.4,168.35,Grand Morgan Waterfall,ville,SEED_1BA6A272DFE2D23F2D74,96579,null,null,2026-07-21,2026,7,21
"3837 Andrew Skyway Apt. 076, Saint-Étienne, France",966,Saint-Étienne,France,2026-07-20T18:41:31.235034Z,Ruizside,false,827,45.43389,4.39,Royal Schultz Square,null,SEED_4A6658CE5D6E27DC81AB,57166,null,null,2026-07-21,2026,7,21
"350 Matthew Union Suite 903, The Hague, Netherlands",null,The Hague,Netherlands,2026-07-20T18:41:31.351411Z,Seanland,false,828,52.07667,4.29861,Old Berry Waterfall,null,SEED_D8B411EF846C2B1DB6E7,87758,null,null,2026-07-21,2026,7,21


## Removing Duplicates

In [0]:
df = df.dropDuplicates(subset=["id"])


### Dividning New vs Old Records

In [0]:
if init_load_flag == 0:
    df_old = spark.sql('''select DimGoogleAddressKey, id, create_date, update_date from travel_journal_catalog.gold.DimGoogleAddress''')
    

else:
    df_old = spark.sql('''select 0 DimGoogleAddressKey, 0 id, 0 create_date, 0 update_date from travel_journal_catalog.silver.google_maps_address_silver where 1=0''')

In [0]:
df_old.display()

DimGoogleAddressKey,id,create_date,update_date
1,9,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
2,820,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
3,821,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
4,822,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
5,823,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
6,824,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
7,825,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
8,826,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
9,827,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
10,828,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z


### Renaming Columns of df_old

In [0]:
df_old = df_old.withColumnRenamed("DimGoogleAddressKey", "old_DimGoogleAddressKey")\
    .withColumnRenamed("id","old_id")\
    .withColumnRenamed("create_date","old_create_date")\
    .withColumnRenamed("update_date","old_update_date")


In [0]:
df_old.display()

old_DimGoogleAddressKey,old_id,old_create_date,old_update_date
1,9,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
2,820,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
3,821,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
4,822,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
5,823,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
6,824,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
7,825,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
8,826,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
9,827,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
10,828,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z


## Applying Join with Old Records

In [0]:
from pyspark.sql.functions import col
df_join = df.join(df_old, df.id == df_old.old_id, "left")


In [0]:
df_join.display()


address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,old_DimGoogleAddressKey,old_id,old_create_date,old_update_date
"1-1 Honmaru, Naka Ward, Nagoya, Aichi 460-0031, Japan",1,1,Japan,2026-06-23T21:25:29.26092Z,null,false,9,35.1848636,136.899927,Honmaru Palace,Naka Ward,ChIJleKGf8t2A2ARY_ZkNA5RTo8,460-0031,Aichi,Aichi,2026-06-24,2026,6,24,1,9,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"830 Smith Court, Kristiansand, Norway",85977,Kristiansand,Norway,2026-07-20T18:41:30.405255Z,North Crystal,false,820,58.14671,7.9956,Golden Mccarthy Bazaar,null,SEED_D66A3D33A81D3720244D,null,null,null,2026-07-21,2026,7,21,2,820,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"8551 Cody Fork Suite 501, Valencia, Spain",98738,Valencia,Spain,2026-07-20T18:41:30.526168Z,South Andreamouth,false,821,39.47391,-0.37966,Old Hodges Tower,null,SEED_92F79A98373BC16A5770,53545,null,null,2026-07-21,2026,7,21,3,821,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"18153 Berger Fork, Phnom Penh, Cambodia",null,Phnom Penh,Cambodia,2026-07-20T18:41:30.644969Z,Sarahtown,false,822,11.56245,104.91601,Royal Wade Bazaar,side,SEED_F2E6E20A2A2E3FF0535E,23335,null,null,2026-07-21,2026,7,21,4,822,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"1125 Selena Plaza, Durban, South Africa",null,Durban,South Africa,2026-07-20T18:41:30.765536Z,East Christine,false,823,-29.8579,31.0292,Hidden Peterson Pier,town,SEED_CFFD589432E71AED70A2,64966,null,null,2026-07-21,2026,7,21,5,823,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"614 Hardy Walks, Callao, Peru",null,Callao,Peru,2026-07-20T18:41:30.881142Z,Port Kimberlystad,false,824,-12.05659,-77.11814,Golden May Harbor,view,SEED_E681890FDE3792E1C646,null,null,null,2026-07-21,2026,7,21,6,824,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"042 Johnson Shoals, Budapest XVII. kerület, Hungary",null,Budapest XVII. kerület,Hungary,2026-07-20T18:41:30.995516Z,Angelville,false,825,47.47997,19.25388,Big Bailey Market,furt,SEED_D5CC32B458E07913F8FC,45535,null,null,2026-07-21,2026,7,21,7,825,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"9757 Joanne Heights Apt. 555, Invercargill, New Zealand",357,Invercargill,New Zealand,2026-07-20T18:41:31.113575Z,Owensfurt,false,826,-46.4,168.35,Grand Morgan Waterfall,ville,SEED_1BA6A272DFE2D23F2D74,96579,null,null,2026-07-21,2026,7,21,8,826,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"3837 Andrew Skyway Apt. 076, Saint-Étienne, France",966,Saint-Étienne,France,2026-07-20T18:41:31.235034Z,Ruizside,false,827,45.43389,4.39,Royal Schultz Square,null,SEED_4A6658CE5D6E27DC81AB,57166,null,null,2026-07-21,2026,7,21,9,827,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"350 Matthew Union Suite 903, The Hague, Netherlands",null,The Hague,Netherlands,2026-07-20T18:41:31.351411Z,Seanland,false,828,52.07667,4.29861,Old Berry Waterfall,null,SEED_D8B411EF846C2B1DB6E7,87758,null,null,2026-07-21,2026,7,21,10,828,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z


### Separating New vs Old Records

In [0]:
df_new = df_join.filter(df_join.old_DimGoogleAddressKey.isNull())
df_new.display()


address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,old_DimGoogleAddressKey,old_id,old_create_date,old_update_date


In [0]:
df_old = df_join.filter(df_join.old_DimGoogleAddressKey.isNotNull())
df_old.display()

address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,old_DimGoogleAddressKey,old_id,old_create_date,old_update_date
"1-1 Honmaru, Naka Ward, Nagoya, Aichi 460-0031, Japan",1,1,Japan,2026-06-23T21:25:29.26092Z,null,false,9,35.1848636,136.899927,Honmaru Palace,Naka Ward,ChIJleKGf8t2A2ARY_ZkNA5RTo8,460-0031,Aichi,Aichi,2026-06-24,2026,6,24,1,9,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"830 Smith Court, Kristiansand, Norway",85977,Kristiansand,Norway,2026-07-20T18:41:30.405255Z,North Crystal,false,820,58.14671,7.9956,Golden Mccarthy Bazaar,null,SEED_D66A3D33A81D3720244D,null,null,null,2026-07-21,2026,7,21,2,820,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"8551 Cody Fork Suite 501, Valencia, Spain",98738,Valencia,Spain,2026-07-20T18:41:30.526168Z,South Andreamouth,false,821,39.47391,-0.37966,Old Hodges Tower,null,SEED_92F79A98373BC16A5770,53545,null,null,2026-07-21,2026,7,21,3,821,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"18153 Berger Fork, Phnom Penh, Cambodia",null,Phnom Penh,Cambodia,2026-07-20T18:41:30.644969Z,Sarahtown,false,822,11.56245,104.91601,Royal Wade Bazaar,side,SEED_F2E6E20A2A2E3FF0535E,23335,null,null,2026-07-21,2026,7,21,4,822,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"1125 Selena Plaza, Durban, South Africa",null,Durban,South Africa,2026-07-20T18:41:30.765536Z,East Christine,false,823,-29.8579,31.0292,Hidden Peterson Pier,town,SEED_CFFD589432E71AED70A2,64966,null,null,2026-07-21,2026,7,21,5,823,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"614 Hardy Walks, Callao, Peru",null,Callao,Peru,2026-07-20T18:41:30.881142Z,Port Kimberlystad,false,824,-12.05659,-77.11814,Golden May Harbor,view,SEED_E681890FDE3792E1C646,null,null,null,2026-07-21,2026,7,21,6,824,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"042 Johnson Shoals, Budapest XVII. kerület, Hungary",null,Budapest XVII. kerület,Hungary,2026-07-20T18:41:30.995516Z,Angelville,false,825,47.47997,19.25388,Big Bailey Market,furt,SEED_D5CC32B458E07913F8FC,45535,null,null,2026-07-21,2026,7,21,7,825,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"9757 Joanne Heights Apt. 555, Invercargill, New Zealand",357,Invercargill,New Zealand,2026-07-20T18:41:31.113575Z,Owensfurt,false,826,-46.4,168.35,Grand Morgan Waterfall,ville,SEED_1BA6A272DFE2D23F2D74,96579,null,null,2026-07-21,2026,7,21,8,826,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"3837 Andrew Skyway Apt. 076, Saint-Étienne, France",966,Saint-Étienne,France,2026-07-20T18:41:31.235034Z,Ruizside,false,827,45.43389,4.39,Royal Schultz Square,null,SEED_4A6658CE5D6E27DC81AB,57166,null,null,2026-07-21,2026,7,21,9,827,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z
"350 Matthew Union Suite 903, The Hague, Netherlands",null,The Hague,Netherlands,2026-07-20T18:41:31.351411Z,Seanland,false,828,52.07667,4.29861,Old Berry Waterfall,null,SEED_D8B411EF846C2B1DB6E7,87758,null,null,2026-07-21,2026,7,21,10,828,2026-07-31T06:27:46.615752Z,2026-07-31T06:27:46.615752Z


### Prepare df_old

In [0]:
# Dropping all the columns which are not require

df_old = df_old.drop('old_id','old_update_date')

# Renaming "old_create_date column to create_date"
df_old = df_old.withColumnRenamed("old_DimGoogleAddressKey","DimGoogleAddressKey")

df_old = df_old.withColumnRenamed("old_create_date","create_date")
df_old = df_old.withColumnRenamed("old_update_date","update_date")


df_old = df_old.withColumn("create_date",to_timestamp("create_date"))


# Recreating "update_date"
df_old = df_old.withColumn("update_date",current_timestamp())



In [0]:
df_old.display()

address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,DimGoogleAddressKey,create_date,update_date
"1-1 Honmaru, Naka Ward, Nagoya, Aichi 460-0031, Japan",1,1,Japan,2026-06-23T21:25:29.26092Z,null,false,9,35.1848636,136.899927,Honmaru Palace,Naka Ward,ChIJleKGf8t2A2ARY_ZkNA5RTo8,460-0031,Aichi,Aichi,2026-06-24,2026,6,24,1,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z
"830 Smith Court, Kristiansand, Norway",85977,Kristiansand,Norway,2026-07-20T18:41:30.405255Z,North Crystal,false,820,58.14671,7.9956,Golden Mccarthy Bazaar,null,SEED_D66A3D33A81D3720244D,null,null,null,2026-07-21,2026,7,21,2,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z
"8551 Cody Fork Suite 501, Valencia, Spain",98738,Valencia,Spain,2026-07-20T18:41:30.526168Z,South Andreamouth,false,821,39.47391,-0.37966,Old Hodges Tower,null,SEED_92F79A98373BC16A5770,53545,null,null,2026-07-21,2026,7,21,3,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z
"18153 Berger Fork, Phnom Penh, Cambodia",null,Phnom Penh,Cambodia,2026-07-20T18:41:30.644969Z,Sarahtown,false,822,11.56245,104.91601,Royal Wade Bazaar,side,SEED_F2E6E20A2A2E3FF0535E,23335,null,null,2026-07-21,2026,7,21,4,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z
"1125 Selena Plaza, Durban, South Africa",null,Durban,South Africa,2026-07-20T18:41:30.765536Z,East Christine,false,823,-29.8579,31.0292,Hidden Peterson Pier,town,SEED_CFFD589432E71AED70A2,64966,null,null,2026-07-21,2026,7,21,5,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z
"614 Hardy Walks, Callao, Peru",null,Callao,Peru,2026-07-20T18:41:30.881142Z,Port Kimberlystad,false,824,-12.05659,-77.11814,Golden May Harbor,view,SEED_E681890FDE3792E1C646,null,null,null,2026-07-21,2026,7,21,6,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z
"042 Johnson Shoals, Budapest XVII. kerület, Hungary",null,Budapest XVII. kerület,Hungary,2026-07-20T18:41:30.995516Z,Angelville,false,825,47.47997,19.25388,Big Bailey Market,furt,SEED_D5CC32B458E07913F8FC,45535,null,null,2026-07-21,2026,7,21,7,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z
"9757 Joanne Heights Apt. 555, Invercargill, New Zealand",357,Invercargill,New Zealand,2026-07-20T18:41:31.113575Z,Owensfurt,false,826,-46.4,168.35,Grand Morgan Waterfall,ville,SEED_1BA6A272DFE2D23F2D74,96579,null,null,2026-07-21,2026,7,21,8,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z
"3837 Andrew Skyway Apt. 076, Saint-Étienne, France",966,Saint-Étienne,France,2026-07-20T18:41:31.235034Z,Ruizside,false,827,45.43389,4.39,Royal Schultz Square,null,SEED_4A6658CE5D6E27DC81AB,57166,null,null,2026-07-21,2026,7,21,9,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z
"350 Matthew Union Suite 903, The Hague, Netherlands",null,The Hague,Netherlands,2026-07-20T18:41:31.351411Z,Seanland,false,828,52.07667,4.29861,Old Berry Waterfall,null,SEED_D8B411EF846C2B1DB6E7,87758,null,null,2026-07-21,2026,7,21,10,2026-07-31T06:27:46.615752Z,2026-07-31T06:28:09.955951Z


### Prepare New df

In [0]:
df_new.display()

address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,old_DimGoogleAddressKey,old_id,old_create_date,old_update_date


In [0]:
# Dropping all the columns which are not require

df_new = df_new.drop('old_DimGoogleAddressKey','old_id','old_update_date','old_create_date')


# Recreating "update_date", "create_date" colums with current timestamp
df_new = df_new.withColumn("update_date",current_timestamp())
df_new = df_new.withColumn("create_date",current_timestamp())



In [0]:
df_new.display()

address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,update_date,create_date


## Surrogate Key - From 1 

In [0]:
df_new = df_new.withColumn("DimGoogleAddressKey",monotonically_increasing_id()+lit(1))

In [0]:
df_new.display()

address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,update_date,create_date,DimGoogleAddressKey


### Adding Max Surrogatekey

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0

else:
    df_maxsurrogate = spark.sql('''select max(DimGoogleAddressKey) as max_surrogate_key from travel_journal_catalog.gold.DimGoogleAddress''')
    #Converting df_maxsur to max_surrogate_key variable
    max_surrogate_key = df_maxsurrogate.collect()[0]['max_surrogate_key']

In [0]:
print(max_surrogate_key)

219


In [0]:
df_new = df_new.withColumn("DimGoogleAddressKey", lit(max_surrogate_key)+col("DimGoogleAddressKey"))

In [0]:
df_new.display()

address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,update_date,create_date,DimGoogleAddressKey


## Union of df_old and df_new

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.display()

address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,update_date,create_date,DimGoogleAddressKey
"1-1 Honmaru, Naka Ward, Nagoya, Aichi 460-0031, Japan",1,1,Japan,2026-06-23T21:25:29.26092Z,null,false,9,35.1848636,136.899927,Honmaru Palace,Naka Ward,ChIJleKGf8t2A2ARY_ZkNA5RTo8,460-0031,Aichi,Aichi,2026-06-24,2026,6,24,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,1
"830 Smith Court, Kristiansand, Norway",85977,Kristiansand,Norway,2026-07-20T18:41:30.405255Z,North Crystal,false,820,58.14671,7.9956,Golden Mccarthy Bazaar,null,SEED_D66A3D33A81D3720244D,null,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,2
"8551 Cody Fork Suite 501, Valencia, Spain",98738,Valencia,Spain,2026-07-20T18:41:30.526168Z,South Andreamouth,false,821,39.47391,-0.37966,Old Hodges Tower,null,SEED_92F79A98373BC16A5770,53545,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,3
"18153 Berger Fork, Phnom Penh, Cambodia",null,Phnom Penh,Cambodia,2026-07-20T18:41:30.644969Z,Sarahtown,false,822,11.56245,104.91601,Royal Wade Bazaar,side,SEED_F2E6E20A2A2E3FF0535E,23335,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,4
"1125 Selena Plaza, Durban, South Africa",null,Durban,South Africa,2026-07-20T18:41:30.765536Z,East Christine,false,823,-29.8579,31.0292,Hidden Peterson Pier,town,SEED_CFFD589432E71AED70A2,64966,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,5
"614 Hardy Walks, Callao, Peru",null,Callao,Peru,2026-07-20T18:41:30.881142Z,Port Kimberlystad,false,824,-12.05659,-77.11814,Golden May Harbor,view,SEED_E681890FDE3792E1C646,null,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,6
"042 Johnson Shoals, Budapest XVII. kerület, Hungary",null,Budapest XVII. kerület,Hungary,2026-07-20T18:41:30.995516Z,Angelville,false,825,47.47997,19.25388,Big Bailey Market,furt,SEED_D5CC32B458E07913F8FC,45535,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,7
"9757 Joanne Heights Apt. 555, Invercargill, New Zealand",357,Invercargill,New Zealand,2026-07-20T18:41:31.113575Z,Owensfurt,false,826,-46.4,168.35,Grand Morgan Waterfall,ville,SEED_1BA6A272DFE2D23F2D74,96579,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,8
"3837 Andrew Skyway Apt. 076, Saint-Étienne, France",966,Saint-Étienne,France,2026-07-20T18:41:31.235034Z,Ruizside,false,827,45.43389,4.39,Royal Schultz Square,null,SEED_4A6658CE5D6E27DC81AB,57166,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,9
"350 Matthew Union Suite 903, The Hague, Netherlands",null,The Hague,Netherlands,2026-07-20T18:41:31.351411Z,Seanland,false,828,52.07667,4.29861,Old Berry Waterfall,null,SEED_D8B411EF846C2B1DB6E7,87758,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:17.106873Z,2026-07-31T06:27:46.615752Z,10


### SCD Type 1

In [0]:
from delta.tables import DeltaTable

Create Upsert Conditional


In [0]:
if spark.catalog.tableExists("travel_journal_catalog.gold.DimGoogleAddress"):
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@databricktraveljournal.dfs.core.windows.net/DimGoogleAddress")

    dlt_obj.alias("trg").merge(
        df_final.alias("src"),"trg.DimGoogleAddressKey = src.DimGoogleAddressKey")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
        
else:
    df_final.write.mode("overwrite")\
    .format("delta")\
    .option("path", "abfss://gold@databricktraveljournal.dfs.core.windows.net/DimGoogleAddress")\
    .saveAsTable("travel_journal_catalog.gold.DimGoogleAddress")

    

In [0]:
%sql

SELECT * FROM travel_journal_catalog.gold.DimGoogleAddress

address,building,city,country,created_at,district,flag,id,latitude,longitude,name,neighborhood,place_id,postal_code,state_province,street,date_type,year,month,day,update_date,create_date,DimGoogleAddressKey
"1-1 Honmaru, Naka Ward, Nagoya, Aichi 460-0031, Japan",1,1,Japan,2026-06-23T21:25:29.26092Z,null,false,9,35.1848636,136.899927,Honmaru Palace,Naka Ward,ChIJleKGf8t2A2ARY_ZkNA5RTo8,460-0031,Aichi,Aichi,2026-06-24,2026,6,24,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,1
"830 Smith Court, Kristiansand, Norway",85977,Kristiansand,Norway,2026-07-20T18:41:30.405255Z,North Crystal,false,820,58.14671,7.9956,Golden Mccarthy Bazaar,null,SEED_D66A3D33A81D3720244D,null,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,2
"8551 Cody Fork Suite 501, Valencia, Spain",98738,Valencia,Spain,2026-07-20T18:41:30.526168Z,South Andreamouth,false,821,39.47391,-0.37966,Old Hodges Tower,null,SEED_92F79A98373BC16A5770,53545,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,3
"18153 Berger Fork, Phnom Penh, Cambodia",null,Phnom Penh,Cambodia,2026-07-20T18:41:30.644969Z,Sarahtown,false,822,11.56245,104.91601,Royal Wade Bazaar,side,SEED_F2E6E20A2A2E3FF0535E,23335,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,4
"1125 Selena Plaza, Durban, South Africa",null,Durban,South Africa,2026-07-20T18:41:30.765536Z,East Christine,false,823,-29.8579,31.0292,Hidden Peterson Pier,town,SEED_CFFD589432E71AED70A2,64966,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,5
"614 Hardy Walks, Callao, Peru",null,Callao,Peru,2026-07-20T18:41:30.881142Z,Port Kimberlystad,false,824,-12.05659,-77.11814,Golden May Harbor,view,SEED_E681890FDE3792E1C646,null,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,6
"042 Johnson Shoals, Budapest XVII. kerület, Hungary",null,Budapest XVII. kerület,Hungary,2026-07-20T18:41:30.995516Z,Angelville,false,825,47.47997,19.25388,Big Bailey Market,furt,SEED_D5CC32B458E07913F8FC,45535,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,7
"9757 Joanne Heights Apt. 555, Invercargill, New Zealand",357,Invercargill,New Zealand,2026-07-20T18:41:31.113575Z,Owensfurt,false,826,-46.4,168.35,Grand Morgan Waterfall,ville,SEED_1BA6A272DFE2D23F2D74,96579,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,8
"3837 Andrew Skyway Apt. 076, Saint-Étienne, France",966,Saint-Étienne,France,2026-07-20T18:41:31.235034Z,Ruizside,false,827,45.43389,4.39,Royal Schultz Square,null,SEED_4A6658CE5D6E27DC81AB,57166,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,9
"350 Matthew Union Suite 903, The Hague, Netherlands",null,The Hague,Netherlands,2026-07-20T18:41:31.351411Z,Seanland,false,828,52.07667,4.29861,Old Berry Waterfall,null,SEED_D8B411EF846C2B1DB6E7,87758,null,null,2026-07-21,2026,7,21,2026-07-31T06:28:19.693731Z,2026-07-31T06:27:46.615752Z,10


In [0]:
%sql DESCRIBE CATALOG EXTENDED travel_journal_catalog;

info_name,info_value
Catalog Name,travel_journal_catalog
Comment,
Owner,siwale.kama@outlook.com
Catalog Type,Regular
Created By,siwale.kama@outlook.com
Created At,2026-07-14 AD at 06:15:40 UTC
Updated By,siwale.kama@outlook.com
Updated At,2026-08-01 AD at 05:04:32 UTC
Storage Root,abfss://gold@databricktraveljournal.dfs.core.windows.net/catalog_root
Storage Location,abfss://gold@databricktraveljournal.dfs.core.windows.net/catalog_root/__unitystorage/catalogs/ea9cc193-90dd-431f-a3f2-f67e2d7f2de8


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS travel_journal_catalog.gold
MANAGED LOCATION 'abfss://gold@databricktraveljournal.dfs.core.windows.net/managed/';